# Hall of Mirrors

**Jane Street puzzle, April 2015** — [puzzle page](https://www.janestreet.com/puzzles/hall-of-mirrors-index/)

## Puzzle

![](https://www.janestreet.com/puzzles/hall_of_mirrors_alt_20230213.PNG)

Alongside the grid above there are several arrows. Red arrows indicate a laser being shone in the given direction, from just off the edge of the grid. Blue arrows indicate a goal. Your job is to add diagonal mirrors to the grid so that as many lasers reach their goal as possible. A mirror cuts across one or more cells at a 45-degree angle (a cell can contain at most one mirror).

You receive **5 points** for every incoming laser that gets redirected to a goal, and you lose **(_N_+2) points** for every mirror you install, where _N_ is the length of the mirror. (Mirrors may cross over each other. E.g. you could have an "X" shape using two length-2 mirrors, which would count as two length-2 mirrors and not as one length-4 mirror.)

**What's the highest score you can achieve?**

## Solution

_Not started yet._

## Setup

Rows are numbered 0–15 top to bottom, columns 0–15 left to right. The arrow positions below were
read off the puzzle image programmatically, by clustering the red and blue pixels against the
detected grid lines. (The doubled blue arrows in the picture are just the colorblind-friendly
rendering of a single goal arrow — every blue arrow is doubled, every red one single.)

Following the usual method, `score(grid)` recomputes the score of a candidate layout by plain
simulation, completely independent of the CP-SAT model, so the model's answer can be checked
against it. A grid is a 16×16 list of lists holding `"."`, `"/"`, or `"\"`.

In [ ]:
GRID_SIZE = 16

# Directions a beam can travel, written as (row_step, column_step).
# Rows are numbered 0-15 top to bottom, columns 0-15 left to right.
UP = (-1, 0)
DOWN = (1, 0)
LEFT = (0, -1)
RIGHT = (0, 1)

# How each kind of mirror bends a beam.  A "/" mirror runs from the bottom-left
# corner of its cell to the top-right, so a beam moving right leaves moving up,
# a beam moving down leaves moving left, and so on.  Both maps are their own
# inverse: bouncing is reversible.
SLASH_BOUNCE = {RIGHT: UP, UP: RIGHT, DOWN: LEFT, LEFT: DOWN}
BACKSLASH_BOUNCE = {RIGHT: DOWN, DOWN: RIGHT, UP: LEFT, LEFT: UP}

# Red arrows: lasers shining into the grid.  ("top", 6) is the laser above
# column 6, shining down; ("right", 2) is the laser beside row 2, shining left.
LASERS = [
    ("top", 1), ("top", 6), ("top", 9), ("top", 14),
    ("bottom", 1), ("bottom", 5), ("bottom", 6),
    ("bottom", 9), ("bottom", 10), ("bottom", 14),
    ("left", 1), ("left", 3), ("left", 5), ("left", 11),
    ("right", 1), ("right", 2), ("right", 3), ("right", 4), ("right", 5),
    ("right", 6), ("right", 9), ("right", 10), ("right", 11), ("right", 12),
]

# Blue arrows: goals.  ("left", 7) is hit by a beam that exits the grid from
# row 7 moving left.
GOALS = [
    ("top", 3), ("top", 12),
    ("bottom", 0), ("bottom", 2), ("bottom", 3), ("bottom", 4), ("bottom", 7),
    ("bottom", 8), ("bottom", 11), ("bottom", 12), ("bottom", 13), ("bottom", 15),
    ("left", 7), ("left", 8), ("left", 14),
    ("right", 0), ("right", 7), ("right", 8),
    ("right", 13), ("right", 14), ("right", 15),
]


def laser_start(side, index):
    """Return (cell, direction): where this laser enters the grid, and its heading."""
    if side == "top":
        return (0, index), DOWN
    if side == "bottom":
        return (GRID_SIZE - 1, index), UP
    if side == "left":
        return (index, 0), RIGHT
    if side == "right":
        return (index, GRID_SIZE - 1), LEFT
    raise ValueError(f"unknown side: {side}")


def goal_exit(side, index):
    """Return (cell, direction): the last cell and heading a beam must have to hit this goal."""
    if side == "top":
        return (0, index), UP
    if side == "bottom":
        return (GRID_SIZE - 1, index), DOWN
    if side == "left":
        return (index, 0), LEFT
    if side == "right":
        return (index, GRID_SIZE - 1), RIGHT
    raise ValueError(f"unknown side: {side}")

In [ ]:
def empty_grid():
    """Return a fresh 16x16 grid with no mirrors; "." means an empty cell."""
    grid = []
    for _ in range(GRID_SIZE):
        grid.append(["."] * GRID_SIZE)
    return grid


def trace_beam(grid, cell, direction):
    """Follow a beam through the grid; return the (cell, direction) it exits with.

    `cell` is the first cell the beam enters and `direction` is its heading.
    The result is the last cell the beam occupies and its heading as it steps
    off the grid, in the same format `goal_exit` uses.
    """
    # Because bouncing is reversible, a beam that comes in from outside can
    # never loop forever -- it must exit.  The step cap only guards against
    # bugs in this function.
    for _ in range(4 * GRID_SIZE * GRID_SIZE):
        row, column = cell
        if grid[row][column] == "/":
            direction = SLASH_BOUNCE[direction]
        elif grid[row][column] == "\\":
            direction = BACKSLASH_BOUNCE[direction]
        next_row = row + direction[0]
        next_column = column + direction[1]
        beam_exits = (next_row < 0 or next_row >= GRID_SIZE
                      or next_column < 0 or next_column >= GRID_SIZE)
        if beam_exits:
            return cell, direction
        cell = (next_row, next_column)
    raise RuntimeError("beam never exited -- trace_beam has a bug")


def mirror_lengths(grid):
    """Return a list with the length of every mirror on the grid.

    A mirror is a maximal diagonal run of cells with the same orientation:
    "\\" cells chain toward the lower-right, "/" cells toward the lower-left.
    (Splitting a run into shorter mirrors is never worth it: same cells
    covered, but the +2 is paid more than once, so we treat runs as mirrors.)
    """
    lengths = []
    for row in range(GRID_SIZE):
        for column in range(GRID_SIZE):
            if grid[row][column] == "\\":
                # count each run once, from its top end only
                continues_a_run = (row > 0 and column > 0
                                   and grid[row - 1][column - 1] == "\\")
                if not continues_a_run:
                    length = 0
                    walk_row, walk_column = row, column
                    while (walk_row < GRID_SIZE and walk_column < GRID_SIZE
                           and grid[walk_row][walk_column] == "\\"):
                        length += 1
                        walk_row += 1
                        walk_column += 1
                    lengths.append(length)
            if grid[row][column] == "/":
                continues_a_run = (row > 0 and column < GRID_SIZE - 1
                                   and grid[row - 1][column + 1] == "/")
                if not continues_a_run:
                    length = 0
                    walk_row, walk_column = row, column
                    while (walk_row < GRID_SIZE and walk_column >= 0
                           and grid[walk_row][walk_column] == "/"):
                        length += 1
                        walk_row += 1
                        walk_column -= 1
                    lengths.append(length)
    return lengths


def mirror_cost(grid):
    """Return the total points paid for mirrors: (length + 2) for each one."""
    total = 0
    for length in mirror_lengths(grid):
        total += length + 2
    return total


def lasers_reaching_goals(grid):
    """Return how many lasers exit the grid exactly through a goal arrow."""
    goal_exits = set()
    for side, index in GOALS:
        goal_exits.add(goal_exit(side, index))
    count = 0
    for side, index in LASERS:
        start_cell, start_direction = laser_start(side, index)
        if trace_beam(grid, start_cell, start_direction) in goal_exits:
            count += 1
    return count


def score(grid):
    """Return the puzzle score of this grid: 5 per laser on a goal, minus mirror costs."""
    return 5 * lasers_reaching_goals(grid) - mirror_cost(grid)

In [ ]:
def show(grid):
    """Print the grid with its border arrows, oriented like the puzzle picture.

    Arrows point the way the picture's arrows do: a laser arrow points into
    the grid, a goal arrow points out of it.
    """
    arrow_for = {
        ("top", "laser"): "v", ("top", "goal"): "^",
        ("bottom", "laser"): "^", ("bottom", "goal"): "v",
        ("left", "laser"): ">", ("left", "goal"): "<",
        ("right", "laser"): "<", ("right", "goal"): ">",
    }
    marks = {"top": {}, "bottom": {}, "left": {}, "right": {}}
    for side, index in LASERS:
        marks[side][index] = arrow_for[(side, "laser")]
    for side, index in GOALS:
        marks[side][index] = arrow_for[(side, "goal")]

    top_row = []
    bottom_row = []
    for column in range(GRID_SIZE):
        top_row.append(marks["top"].get(column, " "))
        bottom_row.append(marks["bottom"].get(column, " "))
    print("  " + " ".join(top_row))
    for row in range(GRID_SIZE):
        left_mark = marks["left"].get(row, " ")
        right_mark = marks["right"].get(row, " ")
        print(left_mark + " " + " ".join(grid[row]) + " " + right_mark)
    print("  " + " ".join(bottom_row))


show(empty_grid())

In [ ]:
# The empty grid scores 0: no mirrors to pay for, and no laser happens to fly
# straight into a goal (every straight shot lands on a red slot or a blank edge).
assert score(empty_grid()) == 0

# A single "/" at row 7, column 1 redirects two lasers at once: the laser above
# column 1 comes down and bounces left into the goal ("left", 7), and the laser
# below column 1 comes up the same column and bounces right into ("right", 7).
test_grid = empty_grid()
test_grid[7][1] = "/"
assert trace_beam(test_grid, *laser_start("top", 1)) == goal_exit("left", 7)
assert trace_beam(test_grid, *laser_start("bottom", 1)) == goal_exit("right", 7)
assert score(test_grid) == 2 * 5 - (1 + 2)

# A single "\" at row 1, column 4: the laser entering row 1 from the left
# bounces down and exits below column 4 -- the goal ("bottom", 4).  The laser
# entering row 1 from the right bounces up to a blank edge slot and scores
# nothing.
test_grid = empty_grid()
test_grid[1][4] = "\\"
assert trace_beam(test_grid, *laser_start("left", 1)) == goal_exit("bottom", 4)
assert trace_beam(test_grid, *laser_start("right", 1)) == ((0, 4), UP)
assert score(test_grid) == 5 - (1 + 2)

# Mirror accounting: an X made of two length-2 mirrors that cross, plus a
# separate length-3 mirror.  The X counts as two mirrors, not one length-4.
test_grid = empty_grid()
test_grid[0][0] = "\\"
test_grid[1][1] = "\\"
test_grid[0][1] = "/"
test_grid[1][0] = "/"
test_grid[5][5] = "\\"
test_grid[6][6] = "\\"
test_grid[7][7] = "\\"
assert sorted(mirror_lengths(test_grid)) == [2, 2, 3]
assert mirror_cost(test_grid) == (2 + 2) + (2 + 2) + (3 + 2)

print("all sanity checks passed")

## CP-SAT model

Booleans per cell say whether it holds a `/` mirror, a `\` mirror, or neither. Beams are encoded
as an occupancy circuit: for every cell and direction, `arrives` says a beam is in the cell moving
that way before any bounce, and `leaves` says the same after the bounce. Each `leaves` literal is
the sum of three AND-terms (went straight through an empty cell, or bounced off one of the two
mirror kinds), and each `arrives` equals the neighboring cell's `leaves` — with border arrivals
pinned to 1 where a red laser shines in and 0 everywhere else. A goal is hit exactly when its
border cell's outward `leaves` literal is true.

Because bouncing is reversible, two beams can never travel the same cell in the same direction,
so one boolean per (cell, direction) covers all 24 lasers at once. Closed loops of light
disconnected from every laser also satisfy the equations, but a loop can never reach the border,
so it cannot affect the score.

Mirrors are counted at their top ends: a mirror cell starts a new mirror exactly when the
diagonally previous cell does not continue the run. The objective is then

$$\text{score} = 5 \cdot \text{goals hit} - \text{mirror cells} - 2 \cdot \text{mirror count},$$

which is the same as paying $$\text{length} + 2$$ per mirror.

In [ ]:
from ortools.sat.python import cp_model

DIRECTIONS = [UP, DOWN, LEFT, RIGHT]


def add_and_equality(model, result, literal_a, literal_b):
    """Constrain result == (literal_a AND literal_b), in both directions."""
    model.add_implication(result, literal_a)
    model.add_implication(result, literal_b)
    # the remaining direction, "(a and b) implies result", written as a clause
    model.add_bool_or([literal_a.negated(), literal_b.negated(), result])


def build_model():
    """Build the CP-SAT model; return (model, slash, backslash, total_score).

    `slash` and `backslash` map (row, column) to the booleans choosing that
    cell's mirror; `total_score` is the integer variable the model maximizes.
    """
    model = cp_model.CpModel()

    # --- the mirror layout ------------------------------------------------
    slash = {}
    backslash = {}
    no_mirror = {}
    mirror_cell_literals = []
    for row in range(GRID_SIZE):
        for column in range(GRID_SIZE):
            slash[row, column] = model.new_bool_var(f"slash_r{row}_c{column}")
            backslash[row, column] = model.new_bool_var(f"backslash_r{row}_c{column}")
            no_mirror[row, column] = model.new_bool_var(f"empty_r{row}_c{column}")
            # a cell holds one mirror or none
            model.add_exactly_one(
                [slash[row, column], backslash[row, column], no_mirror[row, column]])
            mirror_cell_literals.append(slash[row, column])
            mirror_cell_literals.append(backslash[row, column])

    # --- beam occupancy ---------------------------------------------------
    # arrives[cell, direction]: a beam is in this cell moving `direction`,
    # before the cell's mirror (if any) bends it.  leaves[...] is the same
    # beam after the bend.  Beams pass through each other freely, so several
    # of these can be true at once in one cell.
    arrives = {}
    leaves = {}
    for row in range(GRID_SIZE):
        for column in range(GRID_SIZE):
            for direction in DIRECTIONS:
                name = f"r{row}_c{column}_d{direction[0]}_{direction[1]}"
                arrives[row, column, direction] = model.new_bool_var("arrives_" + name)
                leaves[row, column, direction] = model.new_bool_var("leaves_" + name)

    # what leaves a cell is what arrived, bent by the cell's content
    for row in range(GRID_SIZE):
        for column in range(GRID_SIZE):
            for direction_out in DIRECTIONS:
                name = f"r{row}_c{column}_d{direction_out[0]}_{direction_out[1]}"
                # Three mutually exclusive ways to leave in this direction.
                # The bounce maps are their own inverse, so SLASH_BOUNCE[out]
                # is also the heading a beam must ARRIVE with to leave via
                # `out` after bouncing off a "/".
                straight = model.new_bool_var("straight_" + name)
                add_and_equality(model, straight,
                                 arrives[row, column, direction_out],
                                 no_mirror[row, column])
                bounced_off_slash = model.new_bool_var("off_slash_" + name)
                add_and_equality(model, bounced_off_slash,
                                 arrives[row, column, SLASH_BOUNCE[direction_out]],
                                 slash[row, column])
                bounced_off_backslash = model.new_bool_var("off_backslash_" + name)
                add_and_equality(model, bounced_off_backslash,
                                 arrives[row, column, BACKSLASH_BOUNCE[direction_out]],
                                 backslash[row, column])
                model.add(leaves[row, column, direction_out]
                          == straight + bounced_off_slash + bounced_off_backslash)

    # --- stitch neighboring cells together --------------------------------
    # A beam arriving here moving right is exactly a beam that left the cell
    # on our left moving right.  On the border, arrivals are fixed: 1 where a
    # red laser shines in, 0 everywhere else (nothing else enters the grid).
    laser_entries = set()
    for side, index in LASERS:
        laser_entries.add(laser_start(side, index))

    for row in range(GRID_SIZE):
        for column in range(GRID_SIZE):
            for direction in DIRECTIONS:
                previous_row = row - direction[0]
                previous_column = column - direction[1]
                previous_is_inside = (0 <= previous_row < GRID_SIZE
                                      and 0 <= previous_column < GRID_SIZE)
                if previous_is_inside:
                    model.add(arrives[row, column, direction]
                              == leaves[previous_row, previous_column, direction])
                elif ((row, column), direction) in laser_entries:
                    model.add(arrives[row, column, direction] == 1)
                else:
                    model.add(arrives[row, column, direction] == 0)

    # --- goals ------------------------------------------------------------
    # a goal is hit exactly when a beam leaves its border cell heading out
    goal_hit_literals = []
    for side, index in GOALS:
        (goal_row, goal_column), direction_out = goal_exit(side, index)
        goal_hit_literals.append(leaves[goal_row, goal_column, direction_out])

    # --- mirror count, for the +2 per mirror ------------------------------
    # count each mirror at its top end: a mirror cell starts a new mirror
    # exactly when the diagonally previous cell does not continue the run
    mirror_start_literals = []
    for row in range(GRID_SIZE):
        for column in range(GRID_SIZE):
            starts_backslash = model.new_bool_var(f"starts_backslash_r{row}_c{column}")
            previous_cell_exists = row > 0 and column > 0
            if previous_cell_exists:
                add_and_equality(model, starts_backslash,
                                 backslash[row, column],
                                 backslash[row - 1, column - 1].negated())
            else:
                model.add(starts_backslash == backslash[row, column])
            mirror_start_literals.append(starts_backslash)

            starts_slash = model.new_bool_var(f"starts_slash_r{row}_c{column}")
            previous_cell_exists = row > 0 and column < GRID_SIZE - 1
            if previous_cell_exists:
                add_and_equality(model, starts_slash,
                                 slash[row, column],
                                 slash[row - 1, column + 1].negated())
            else:
                model.add(starts_slash == slash[row, column])
            mirror_start_literals.append(starts_slash)

    # --- objective --------------------------------------------------------
    # each mirror costs (length + 2): one point per covered cell, two per
    # mirror.  Filling every cell with length-1 mirrors would cost 3 points
    # per cell, hence the lower bound on the score.
    worst_possible_score = -(3 * GRID_SIZE * GRID_SIZE)
    best_possible_score = 5 * len(LASERS)
    total_score = model.new_int_var(worst_possible_score, best_possible_score, "total_score")
    model.add(total_score == 5 * sum(goal_hit_literals)
              - sum(mirror_cell_literals) - 2 * sum(mirror_start_literals))
    model.maximize(total_score)

    return model, slash, backslash, total_score

In [ ]:
model, slash, backslash, total_score = build_model()

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 300.0   # raise this if status comes back FEASIBLE
solver.parameters.num_workers = 8
status = solver.solve(model)

print("status:", solver.status_name(status))
print("score: ", solver.value(total_score))
print("bound: ", solver.best_objective_bound)

In [ ]:
best_grid = empty_grid()
for row in range(GRID_SIZE):
    for column in range(GRID_SIZE):
        if solver.value(slash[row, column]) == 1:
            best_grid[row][column] = "/"
        if solver.value(backslash[row, column]) == 1:
            best_grid[row][column] = "\\"

# the plain-Python scorer must agree with the model's objective
assert score(best_grid) == solver.value(total_score)

print("score", score(best_grid), "with", lasers_reaching_goals(best_grid),
      "lasers on goal and", len(mirror_lengths(best_grid)), "mirrors")
show(best_grid)